# Build the standalone research panel
Run after extraction and quality review. Core fields must be numeric and present; source numeric strings remain unchanged.


In [ ]:
from pathlib import Path
import pandas as pd
from google.colab import drive
from IPython.display import display

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/nse-financial-research')
INPUT = ROOT / 'data/interim/company_quarterly_financials_extracted.csv'
OUTPUT = ROOT / 'data/processed'
START_DATE = '2022-04-01'
END_DATE = '2024-12-31'
CORE_FIELDS = ['PAT', 'TOTAL_INCOME', 'PROFIT_BEFORE_TAX', 'TAX_EXPENSE', 'EPS_BASIC']

d = pd.read_csv(INPUT, dtype=str)
required = ['NSE_SYMBOL', 'PERIOD_START', 'PERIOD_END',
            'STANDALONE_CONSOLIDATED', 'STRICT_QUARTER_CANDIDATE'] + CORE_FIELDS
missing = sorted(set(required) - set(d.columns))
if missing:
    raise ValueError(f'Missing required columns: {missing}')
starts = pd.to_datetime(d.PERIOD_START, errors='coerce')
ends = pd.to_datetime(d.PERIOD_END, errors='coerce')
expected = pd.period_range(START_DATE, END_DATE, freq='Q').end_time.normalize()
mask = (d.STANDALONE_CONSOLIDATED.str.strip().str.lower().eq('standalone')
        & d.STRICT_QUARTER_CANDIDATE.str.lower().eq('true')
        & starts.ge(pd.Timestamp(START_DATE))
        & ends.le(pd.Timestamp(END_DATE))
        & ends.isin(expected)
        & ((ends - starts).dt.days + 1).between(90, 92))
panel = d.loc[mask].copy()
if panel.empty:
    raise ValueError('No matching standalone quarterly records.')
if panel.NSE_SYMBOL.isna().any() or panel.NSE_SYMBOL.str.strip().eq('').any():
    raise ValueError('Missing company symbols must be resolved.')
panel['PERIOD_START'] = starts.loc[mask].dt.strftime('%Y-%m-%d')
panel['PERIOD_END'] = ends.loc[mask].dt.strftime('%Y-%m-%d')
duplicates = panel.duplicated(['NSE_SYMBOL', 'PERIOD_END'], keep=False)
if duplicates.any():
    display(panel.loc[duplicates, ['NSE_SYMBOL', 'PERIOD_END', 'SOURCE_FILE']])
    raise ValueError('Resolve duplicate company-quarter records before proceeding.')
numeric_core = panel[CORE_FIELDS].apply(pd.to_numeric, errors='coerce')
usable = numeric_core.notna().all(axis=1)
counts = panel.loc[usable].groupby('NSE_SYMBOL').PERIOD_END.nunique()
eligible = counts[counts.eq(len(expected))].index
complete = panel[panel.NSE_SYMBOL.isin(eligible)].copy()
coverage = panel.groupby('NSE_SYMBOL').agg(available_quarters=('PERIOD_END', 'nunique'))
coverage['usable_quarters'] = counts.reindex(coverage.index, fill_value=0)
coverage['complete_sample'] = coverage.index.isin(eligible)
OUTPUT.mkdir(parents=True, exist_ok=True)
panel.sort_values(['NSE_SYMBOL', 'PERIOD_END']).to_csv(OUTPUT / 'standalone_available_panel.csv', index=False)
complete.sort_values(['NSE_SYMBOL', 'PERIOD_END']).to_csv(OUTPUT / 'standalone_complete_panel.csv', index=False)
coverage.to_csv(OUTPUT / 'company_coverage.csv')
print(f'Expected quarters: {len(expected)}')
print(f'Companies with available data: {panel.NSE_SYMBOL.nunique():,}')
print(f'Complete companies: {len(eligible):,}; observations: {len(complete):,}')
print(f'Saved to: {OUTPUT}')
display(coverage.head(20))